# Differentiable minilink
## Write `f` once: every gradient, compiled speed, even C

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/showcase_jax.ipynb)

A minilink model is a pure function of its arguments,

$$\dot{x} = f(x, u, t; p),$$

so JAX can trace it once and then differentiate it, compile it, batch it, and even
translate it to C. This notebook runs each of those on one pendulum. Most capabilities
come two ways: the JAX transformation written by hand, and the minilink shortcut that
does the same thing by name. Both give the same numbers.

| You want | Native JAX | minilink |
| --- | --- | --- |
| $\partial f/\partial x$ and $\partial f/\partial u$ | `jax.jacfwd(lambda x: ev.f(x, u, t))(x)` | `plant.jacobian("f", "x", x, u)` |
| The local linear model | the two Jacobians above | `plant.linearize(x, u)` |
| $\partial f/\partial p$ | `jax.jacfwd(lambda p: ev.f_trace_p(x, u, t, p))(params)` | `plant.jacobian("f", "params", x, u)` |
| A family of rollouts | `jax.vmap` over `ev.rk4_integrate_zoh_trace_p` | `ev.rollout_batch(x0s, params=family)` |
| Fast simulation | `jax.jit` of a hand-written loop | `ev.rk4_integrate_zoh(x0, us, t0, dt)` |
| An optimal input sequence | `jax.grad` through the rollout, then descent | `TrajectoryOptimizationPlanner(problem, n_steps=40).solve()` |
| The physics from data | `jax.vmap` and `jax.grad` on an equation error | JAX by hand for now |
| A controller in C | the traced program | `export_system_to_c(law)`, experimental |

Here `ev = plant.compile(backend="jax")` is the compiled evaluator of the plant.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

## 1. One pure function

Every minilink block, plant or controller computes `f` from its arguments only: no state
hidden on the object, no integrator buffer, no cached force. A pure function is what JAX
needs. JAX does not read Python source: it **traces** the function once on placeholder
values, records the array operations, and hands that record to its transformations.
`jax.jit` compiles it, `jax.grad` and `jax.jacfwd` differentiate it, `jax.vmap` maps it
over a batch.

`compile(backend="jax")` runs that trace for a whole system and returns an
**evaluator**: `f` and fixed-step RK4 rollouts, compiled, in 64-bit floats.

The pendulum below is a point mass on a massless rod, so its equation is the textbook
$m \ell^2 \ddot\theta = \tau - m g \ell \sin\theta - d\,\dot\theta$.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

from minilink import Pendulum

pendulum = Pendulum()
pendulum.params["I"] = 0.0  # point mass on a massless rod
pendulum.params["d"] = 0.2  # viscous damping [N m s]
pendulum.inputs["u"].lower_bound = np.array([-5.0])  # torque limits [N m]
pendulum.inputs["u"].upper_bound = np.array([5.0])

x_bar = np.array([0.5, 1.0])  # theta [rad], dtheta [rad/s]
u_bar = np.array([0.0])  # torque [N m]

evaluator = pendulum.compile(backend="jax")  # traces f once
pendulum.f(x_bar, u_bar)

## 2. Linearization, two ways

The Jacobians of `f` with respect to the state and the input are the $A$ and $B$ of the
local model $\delta\dot{x} = A\,\delta x + B\,\delta u$: their eigenvalues give local
stability, and the pair feeds LQR. By hand, `jax.jacfwd` differentiates the compiled `f`
in the argument that varies.

In [ ]:
A_jax = jax.jacfwd(lambda x: evaluator.f(x, u_bar, 0.0))(x_bar)
B_jax = jax.jacfwd(lambda u: evaluator.f(x_bar, u, 0.0))(u_bar)
A_jax

minilink names the derivative instead: `jacobian(of, wrt)` reads $\partial\,\text{of} /
\partial\,\text{wrt}$, exact under JAX and by finite differences otherwise.

In [ ]:
A = pendulum.jacobian("f", "x", x_bar, u_bar)
B = pendulum.jacobian("f", "u", x_bar, u_bar)
np.allclose(A, A_jax), np.allclose(B, B_jax)

`linearize` returns the whole local model as a linear system, ready for eigenvalues,
Bode plots or LQR.

In [ ]:
linear_model = pendulum.linearize(x_bar, u_bar)  # A(), B(), C(), D()
np.linalg.eigvals(linear_model.A())

## 3. Sensitivity to the physics, two ways

The parameters are an argument of `f`, so JAX differentiates with respect to the physics
itself: $\partial f/\partial p$ for every constant in `params`. The evaluator's parametric
tier `f_trace_p(x, u, t, params)` takes the dictionary as a traced argument.

*Under the hood:* a plant picks NumPy or JAX from the type of its state, so the
hand-written version passes `x` and `u` as JAX arrays. The shortcut takes care of that.

In [ ]:
xj, uj = jnp.asarray(x_bar), jnp.asarray(u_bar)  # JAX arrays pick the JAX branch of f
dfdp_jax = jax.jacfwd(lambda p: evaluator.f_trace_p(xj, uj, 0.0, p))(pendulum.params)
dfdp_jax["l"]  # how dx/dt moves with the rod length

In [ ]:
dfdp = pendulum.jacobian("f", "params", x_bar, u_bar)  # one sensitivity per parameter
dfdp

## 4. A family of rollouts, two ways

Purity also buys batching. Seven rod lengths swing from the same start, undamped. By
hand, `jax.vmap` maps a whole RK4 rollout over the lengths.

In [ ]:
lengths = np.linspace(0.5, 2.0, 7)
x_start = jnp.array([1.0, 0.0])  # released at 1 rad, at rest
u_free = jnp.zeros((1200, 1))  # 1200 steps of 5 ms, no torque


def swing(length):
    p = dict(pendulum.params, l=length, d=0.0)
    return evaluator.rk4_integrate_zoh_trace_p(x_start, u_free, 0.0, 0.005, p)


xs_jax = jax.vmap(swing)(jnp.asarray(lengths))
xs_jax.shape  # (lengths, samples, states)

`rollout_batch` does the same from a `params` dictionary whose leaves carry one value
per rollout.

In [ ]:
family = dict(pendulum.params, l=lengths, d=0.0)  # one length per rollout
x0s = np.tile([1.0, 0.0], (len(lengths), 1))
xs = evaluator.rollout_batch(x0s, n_steps=1200, dt=0.005, params=family)
np.allclose(xs, xs_jax)

Plotted against the dimensionless time $t\sqrt{g/\ell}$, the seven swings collapse onto
one curve: the undamped pendulum has a single dimensionless period.

In [ ]:
import matplotlib.pyplot as plt

t = 0.005 * np.arange(1201)
fig, (ax_t, ax_pi) = plt.subplots(1, 2, figsize=(10, 3.4))
for x_l, ell in zip(np.asarray(xs), lengths):
    ax_t.plot(t, x_l[:, 0], label=f"l = {ell:.2f} m")
    ax_pi.plot(t * np.sqrt(pendulum.params["gravity"] / ell), x_l[:, 0])
ax_t.set(xlabel="t [s]", ylabel="theta [rad]")
ax_t.legend(fontsize=7)
ax_pi.set(xlabel=r"$t\,\sqrt{g/\ell}$", title="the same swing in dimensionless time")
plt.show()

## 5. Compiled: fast simulation

Compiling does not make one call to `f` faster: a single evaluation costs a few
microseconds either way. It pays off when a whole loop runs compiled. Below, the same
1000 RK4 steps three ways: stepped in a Python loop on the NumPy evaluator, as one
compiled rollout, and as 1000 compiled rollouts from 1000 initial angles in one call.
The times depend on the machine.

In [ ]:
ev_numpy = pendulum.compile(backend="numpy")
# 1000 initial angles at rest, and 1000 steps of 5 ms with no torque
x0s = np.column_stack([np.linspace(-3.0, 3.0, 1000), np.zeros(1000)])
u_zero = np.zeros((1000, 1))

xs_one = evaluator.rk4_integrate_zoh(x0s[0], u_zero, 0.0, 0.005)  # first call compiles
xs_all = evaluator.rollout_batch(x0s, n_steps=1000, dt=0.005)  # first call compiles

In [ ]:
# One rollout stepped in Python, one compiled rollout, 1000 compiled rollouts
%timeit -n 1 -r 1 ev_numpy.rk4_integrate_zoh(x0s[0], u_zero, 0.0, 0.005)
%timeit -n 1 -r 3 np.asarray(evaluator.rk4_integrate_zoh(x0s[0], u_zero, 0.0, 0.005))
%timeit -n 1 -r 3 np.asarray(evaluator.rollout_batch(x0s, n_steps=1000, dt=0.005))

On the laptop that wrote this notebook, the thousand compiled rollouts finished sooner
than the single rollout stepped in Python.

## 6. An optimal input sequence, two ways

The purity that lets JAX differentiate `f` lets it differentiate a whole rollout built
from `f`. Choose the 40 torques $U$ that bring the hanging pendulum to
$\theta = 2$ rad in 2 s. By hand: gradient descent on
$J(U) = (\theta_N(U) - 2)^2 + 10^{-4} \sum_k u_k^2$, with $\nabla_U J$ from `jax.grad`
through all 40 RK4 steps, and no adjoint derived by hand.

In [ ]:
x_down = jnp.array([0.0, 0.0])  # hanging, at rest


def J(U):
    # The rollout: 40 RK4 steps of 50 ms, differentiable in U
    xs = evaluator.rk4_integrate_zoh_trace(x_down, U, 0.0, 0.05)
    return (xs[-1, 0] - 2.0) ** 2 + 1e-4 * jnp.sum(U**2)


dJ = jax.jit(jax.grad(J))
U = jnp.zeros((40, 1))
for _ in range(1500):
    U = U - 0.6 * dJ(U)  # descent on all 40 torques at once

evaluator.rk4_integrate_zoh_trace(x_down, U, 0.0, 0.05)[-1]  # final state

minilink states the same goal as a `PlanningProblem` (the plant, a cost, a goal and a
horizon) and hands it to a planner, which transcribes it into a nonlinear program and
solves it with the same JAX gradients.

In [ ]:
from minilink import PlanningProblem, QuadraticCost, TrajectoryOptimizationPlanner

goal = np.array([2.0, 0.0])
cost = QuadraticCost.from_system(
    pendulum, Q=np.zeros((2, 2)), R=1e-2 * np.eye(1), S=np.diag([100.0, 1.0]), xbar=goal
)
problem = PlanningProblem(pendulum, x_start=np.zeros(2), x_goal=goal, tf=2.0, cost=cost)
planner = TrajectoryOptimizationPlanner(problem, n_steps=40, compile_backend="jax")
solution = planner.solve()
print(solution)

In [ ]:
pendulum.plot_trajectory(solution.trajectory)

## 7. The physics from data

Sensitivity to parameters makes **system identification** a gradient problem. Log the
pendulum's states and inputs, forget its gravity and damping, and recover them by
gradient descent on the equation error

$$\mathcal{L}(g, d) = \frac{1}{N}\sum_k \big\| f(x_k, u_k, t_k;\, g, d) - \dot{x}_k \big\|^2 .$$

`jax.vmap` evaluates `f` over the whole log at once and `jax.grad` differentiates the
loss. minilink has no identification tool yet, so this one is JAX by hand.

In [ ]:
t_log = 0.02 * jnp.arange(400)
u_log = 0.5 * jnp.sin(0.8 * t_log).reshape(-1, 1)  # a gentle excitation

# The log: one rollout of the true pendulum, and dx/dt at every sample
x_log = evaluator.rk4_integrate_zoh(jnp.array([0.5, 0.0]), u_log, 0.0, 0.02)[:-1]
dx_log = jax.vmap(evaluator.f_trace)(x_log, u_log, t_log)


f_over_log = jax.vmap(evaluator.f_trace_p, in_axes=(0, 0, 0, None))  # every sample


def loss(theta):
    p = dict(pendulum.params, gravity=theta[0], d=theta[1])
    dx_model = f_over_log(x_log, u_log, t_log, p)
    return jnp.mean((dx_model - dx_log) ** 2)


dloss = jax.jit(jax.grad(loss))
theta = jnp.array([7.0, 1.0])  # a wrong first guess for (gravity, damping)
for _ in range(150):
    theta = theta - 2.0 * dloss(theta)

theta  # the truth is (9.81, 0.2)

## 8. Through a neural law

A neural policy is a `System` too, so a closed loop around an untrained network is one
`f`, and `linearize` differentiates through the network and the physics in one trace.

In [ ]:
from minilink import NeuralPolicyController

neural_loop = NeuralPolicyController(pendulum, hidden=(8, 8)) @ pendulum  # untrained
np.linalg.eigvals(neural_loop.linearize(np.zeros(2)).A())  # the loop's poles

## 9. From the trace to C (experimental)

The record JAX makes of a traced function is a small program of array operations. The
experimental `c_export` module translates it into a standalone C function, so a control
law written once in Python can run on a microcontroller. It covers the operations of
simple control laws; here, the impedance law $u = K_p (r - q) - K_d \dot q$.

*Experimental:* `minilink.experimental` ships with the repository, not with the pip
package, and its API may change.

In [ ]:
from minilink import ImpedanceController
from minilink.experimental.c_export import export_system_to_c, load_exported_c

law = ImpedanceController()  # Kp = 10, Kd = 1
c_source = export_system_to_c(law, "impedance_law")
print(c_source)

`load_exported_c` compiles that C with the system compiler (`cc`) and loads it back into
Python, so the C law can be checked against the equation.

In [ ]:
law_c = load_exported_c(c_source, "impedance_law", n_out=1)
law_c([], [1.0, 0.2, -0.5])  # r, q, dq: 10 (1 - 0.2) - 1 (-0.5) = 8.5

## Recap

One `f`, written once in textbook form, gave exact Jacobians and sensitivities, a family
of rollouts in one call, compiled simulation, an optimal input sequence, the physics from
data, a linearization through a neural network, and a control law in C. Each came from
the same trace. The minilink shortcuts name the common ones, and the evaluator's trace
tier (`f_trace`, `f_trace_p`, `rk4_integrate_zoh_trace`, …) stays open for anything you
write by hand.